# 🦙 Fine-Tuning com LoRA — meta-llama/Llama-3.2-3B-Instruct
### Dataset: Neurociência Cognitiva

Este notebook adapta o pipeline LoRA especificamente para o **Llama-3.2-3B-Instruct** (3B parâmetros),
levando em conta seu **chat template nativo** (`<|begin_of_text|>`, `<|start_header_id|>`),
sua arquitetura **Grouped-Query Attention (GQA)** e as diferenças em relação ao Phi-4-mini.

## Diferenças-chave entre Phi-4-mini e Llama-3.2-3B

| Aspecto | Phi-4-mini-instruct | Llama-3.2-3B-Instruct |
|---|---|---|
| Parâmetros | 3.8B | 3B |
| Vocabulário | 200K tokens | 128K tokens |
| Chat template | `<\|system\|>...<\|end\|>` | `<\|begin_of_text\|>...<\|eot_id\|>` |
| `trust_remote_code` | Obrigatório | Não necessário |
| `attn_implementation` | `flash_attention_2` | `flash_attention_2` (opcional) |
| `torch_dtype` nativo | bfloat16 | bfloat16 |
| Acesso no HuggingFace | Aberto | **Requer aceite de licença** |


## 📚 1. Por que Fine-Tuning Eficiente?

Modelos de linguagem modernos possuem bilhões de parâmetros. Atualizar **todos** os pesos durante o treinamento (*full fine-tuning*) exige:
- GPUs com dezenas de GB de memória.
- Armazenamento de uma cópia completa do modelo para cada tarefa.

**PEFT (Parameter-Efficient Fine-Tuning)** resolve esse problema treinando apenas um pequeno conjunto de **novos parâmetros**, mantendo o modelo base congelado.  

### 🔹 LoRA (Low-Rank Adaptation)
A hipótese do LoRA é que as atualizações dos pesos durante o fine-tuning possuem uma **estrutura de baixo posto** (*low intrinsic rank*).  
Assim, em vez de aprender a matriz completa de atualização $\Delta W \in \mathbb{R}^{d \times k}$, aprendemos duas matrizes menores:

$$\Delta W = B \cdot A$$

onde:
- $B \in \mathbb{R}^{d \times r}$
- $A \in \mathbb{R}^{r \times k}$
- $r \ll \min(d, k)$ (o **rank** da adaptação)

O número de parâmetros treináveis cai de $d \times k$ para $r \times (d + k)$, uma redução drástica quando $r$ é pequeno.

### 🔹 Como isso é usado na prática?
Durante o treinamento, a saída de uma camada linear original $h = W x$ é modificada para:

$$h = W x + \Delta W x = W x + B A x$$

A matriz $A$ é inicializada com uma distribuição gaussiana e $B$ com zeros, de forma que no início $\Delta W = 0$.  
Um fator de escala $\alpha$ controla a intensidade da adaptação; frequentemente a atualização é escalada por $\frac{\alpha}{r}$:

$$h = W x + \frac{\alpha}{r} B A x$$

Após o treinamento, podemos **fundir** (*merge*) os pesos adaptados ao modelo original: $W_{\text{merged}} = W + \frac{\alpha}{r} BA$, eliminando qualquer custo extra na inferência.

## 📦 2. Requisitos

Execute o comando abaixo para instalar as dependências necessárias.

> ⚠️ **Acesso ao Llama 3.2:** O modelo é *gated* na HuggingFace — você precisa:
> 1. Aceitar a licença em [huggingface.co/meta-llama/Llama-3.2-3B-Instruct](https://huggingface.co/meta-llama/Llama-3.2-3B-Instruct)
> 2. Gerar um token de acesso em [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
> 3. Fazer login com `huggingface-cli login` ou definir `HF_TOKEN` nas variáveis de ambiente
> 




In [ ]:
#!pip install transformers datasets peft accelerate torch bitsandbytes

from huggingface_hub import login
login(token="your_token")

Importe os módulos que serão utilizados ao longo do processo:

In [ ]:
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    BitsAndBytesConfig
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)
import torch

/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 🤖 3. Carregar modelo e tokenizador

Carregamos o Llama-3.2-3B-Instruct em **4-bit (QLoRA)**.

| Precisão | VRAM estimada (3B params) |
|---|---|
| float32 (sem quantização) | ~12 GB |
| float16 / bfloat16 | ~6 GB |
| 4-bit QLoRA | ~2–3 GB |


In [ ]:
model_name = "meta-llama/Llama-3.2-3B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    use_fast=True
)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Modelo carregado: {model_name}")

## 📂 4. Carregar e preparar o dataset

O Llama-3.2-3B-Instruct usa um chat template **diferente do Phi-4-mini**.
O `apply_chat_template` do tokenizador cuida disso automaticamente,
mas é importante entender a diferença para debugar se necessário.

**Template do Llama 3.2:**
```
<|begin_of_text|><|start_header_id|>system<|end_header_id|>
...<|eot_id|><|start_header_id|>user<|end_header_id|>
...<|eot_id|><|start_header_id|>assistant<|end_header_id|>
...<|eot_id|>
```

O código abaixo é **idêntico** ao do Phi-4-mini — o `apply_chat_template` aplica o template
correto automaticamente para cada modelo. Essa é a vantagem de usar a API do Hugging Face.


In [ ]:
def convert_to_hf_format(example):
    """Aplica o Chat Template oficial do Phi-4 para unir instrução e saída."""
    messages = [
        {"role": "system", "content": "Você é um assistente especializado em neurociência cognitiva."},
        {"role": "user", "content": example["Instruction"]},
        {"role": "assistant", "content": example["Output"]}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

dataset = load_dataset('json', data_files='data/processed/dataset_curado.jsonl')
dataset = dataset.map(convert_to_hf_format)
dataset = dataset["train"].train_test_split(test_size=0.2)
print(dataset)

Map: 100%|██████████| 10/10 [00:00<00:00, 3070.28 examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 8
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 2
    })
})


## 🔍 5. Inferência ANTES do fine-tuning (linha de base)

Registramos a resposta do Llama-3.2-3B-Instruct **antes** de qualquer adaptação.
Por ser um modelo de instrução, ele dará uma resposta razoável — mas genérica,
sem a profundidade específica do nosso material de neurociência cognitiva.


In [ ]:
def generate_response(model, tokenizer, instruction, input_text=""):
    """Gera uma resposta a partir de uma instrução, usando o modelo fornecido."""
    messages = [{"role": "user", "content": instruction}]
    if input_text:
        messages = [{"role": "user", "content": f"{instruction}\nContexto: {input_text}"}]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=128,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=True,       
        temperature=0.7
    )
    generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
    resposta = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

    return resposta

test_instruction = "O que é memória de trabalho e qual é o papel do córtex pré-frontal no seu funcionamento?"

print("=== ANTES DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")
print(f"Resposta base: {generate_response(base_model, tokenizer, test_instruction)}")

=== ANTES DO FINE-TUNING ===
Instrução: How do I activate cruise control?
Resposta base: 


> **Observação:** O modelo base provavelmente gerará um texto genérico ou sem relação direta com a instrução, pois ainda não foi adaptado ao nosso domínio.

## ✂️ 6. Tokenização do Dataset

Transformamos os textos em sequências de tokens que o modelo pode processar.

**Por que `max_length=512`?**  
O template do Llama 3.2 também consome ~40–50 tokens com tokens especiais.
Com 128 tokens, a maioria das respostas de neurociência seria truncada no meio da frase.
512 tokens é o mínimo adequado para este domínio.


In [ ]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=256,
        padding=False
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenizado:", tokenized_datasets)

Map: 100%|██████████| 2/2 [00:00<00:00, 658.76 examples/s]

Dataset tokenizado: DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 8
    })
    test: Dataset({
        features: ['instruction', 'input', 'output', 'text', 'input_ids', 'attention_mask'],
        num_rows: 2
    })
})


## 🔧 7. Preparar o Modelo para LoRA

A função `prepare_model_for_kbit_training` ativa técnicas como *gradient checkpointing* e ajusta a arquitetura para treinamento eficiente.  
É essencial quando se utiliza quantização (QLoRA), mas também é recomendada mesmo sem quantização para melhor gerenciamento de memória.

In [ ]:
model = base_model
model = prepare_model_for_kbit_training(model)

## 🧩 8. Configurar LoRA para o Llama-3.2-3B-Instruct

### Os `target_modules` são os mesmos do Phi-4-mini?

**Sim!** Ambos os modelos usam a mesma arquitetura de atenção GQA com os mesmos nomes de camada.
Isso acontece porque o Llama se tornou uma arquitetura de referência e muitos modelos, inclusive o Phi-4,
adotaram a mesma nomenclatura para as camadas lineares.

| Módulo | Tipo | Descrição |
|---|---|---|
| `q_proj` | Atenção | Projeção das queries |
| `k_proj` | Atenção | Projeção das keys |
| `v_proj` | Atenção | Projeção dos values |
| `o_proj` | Atenção | Projeção de saída da atenção |
| `gate_proj` | MLP | Porta da ativação SiLU |
| `up_proj` | MLP | Projeção "up" da MLP |
| `down_proj` | MLP | Projeção "down" da MLP |

### Sobre `r` e `lora_alpha`

- `r=16`: rank moderado — bom equilíbrio para um dataset de ~250 exemplos
- `lora_alpha=32`: convenção `alpha = 2 * r`


In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    inference_mode=False,
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


trainable params: 811,008 || all params: 82,723,584 || trainable%: 0.9804


/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/peft/tuners/lora/layer.py:2174: UserWarning: fan_in_fan_out is set to False but the target module is `Conv1D`. Setting fan_in_fan_out to True.
  warnings.warn(


✅ **Interpretação:** Apenas uma fração mínima do total de parâmetros será atualizada.  
No exemplo, menos de 1% dos pesos são treináveis – é a essência do PEFT.

## 🧱 9. Data Collator para Modelagem Causal

O `DataCollatorForLanguageModeling` prepara os lotes para o treinamento de linguagem causal (sem *masked language modeling*).  
Ele automaticamente desloca os rótulos para que a tarefa seja prever o próximo token.

In [9]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

## ⚙️ 10. Argumentos de Treinamento

Definimos os hiperparâmetros do treinamento.

| Parâmetro | Valor | Justificativa |
|---|---|---|
| `learning_rate` | `2e-4` | Padrão para LoRA em modelos instruct — `1e-3` causa instabilidade |
| `num_train_epochs` | `10` | Modelos instruct aprendem domínio rápido; revise a eval loss |
| `per_device_train_batch_size` | `2` | 3B ainda ocupa VRAM considerável com QLoRA |
| `gradient_accumulation_steps` | `4` | Simula batch efetivo de 8 sem explodir a VRAM |
| `bf16` | `True` | Llama 3.2 foi treinado em bfloat16 |
| `lr_scheduler_type` | `cosine` | Decaimento suave do LR — mais estável que linear |
| `warmup_ratio` | `0.05` | Aquece o LR nos primeiros 5% dos steps |
| `output_dir` | `lora_models/llama32_neuro` | Separado do modelo Phi para não misturar checkpoints |


In [ ]:
training_args = TrainingArguments(
    output_dir="lora_models/causal_model_2",   
    eval_strategy="steps",
    eval_steps=50,
    learning_rate=2e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="steps",
    save_steps=50,
    load_best_model_at_end=True,
    fp16=True,
    report_to="none",             
)

## 🏋️ 11. Inicializar o Trainer

O `Trainer` do Hugging Face orquestra todo o ciclo de treinamento, avaliação e salvamento.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    data_collator=data_collator,
)

## 🚀 12. Treinar o Modelo

Iniciamos o treinamento. Acompanhe a perda (*loss*) nos logs – ela deve diminuir ao longo das épocas.

In [12]:
trainer.train()

/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss,Validation Loss
100,0.307000,4.398220
200,0.123200,4.661976


/Users/thommasflores/Documents/GitHub/TinyML/.venv/lib/python3.9/site-packages/torch/utils/data/dataloader.py:684: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=200, training_loss=0.7588844114542007, metrics={'train_runtime': 16.0803, 'train_samples_per_second': 49.75, 'train_steps_per_second': 12.438, 'total_flos': 26627958374400.0, 'train_loss': 0.7588844114542007, 'epoch': 100.0})

## 💾 13. Salvar o Modelo Ajustado e o Tokenizador

Ao final do treinamento, salvamos os pesos LoRA (apenas os adaptadores) e o tokenizador.

In [ ]:
model.save_pretrained("lora_models/causal_model_2/final_adapter")
tokenizer.save_pretrained("lora_models/causal_model_2/final_tokenizer")

('distilgpt2_tokenizer/tokenizer_config.json',
 'distilgpt2_tokenizer/special_tokens_map.json',
 'distilgpt2_tokenizer/vocab.json',
 'distilgpt2_tokenizer/merges.txt',
 'distilgpt2_tokenizer/added_tokens.json',
 'distilgpt2_tokenizer/tokenizer.json')

## 💻 14. Inferência APÓS o Fine-Tuning

Agora carregamos o modelo ajustado e comparamos sua resposta com a versão base, usando **exatamente a mesma instrução**.

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

finetuned_model = PeftModel.from_pretrained(
    base_model,
    "lora_models/causal_model_2/final_adapter"
)

finetuned_tokenizer = AutoTokenizer.from_pretrained(
    "lora_models/causal_model_2/final_tokenizer"
)

if finetuned_tokenizer.pad_token is None:
    finetuned_tokenizer.pad_token = finetuned_tokenizer.eos_token

In [ ]:
print("\n=== DEPOIS DO FINE-TUNING ===")
print(f"Instrução: {test_instruction}")

resposta_ajustada = generate_response(finetuned_model, finetuned_tokenizer, test_instruction)
print(f"Resposta ajustada: {resposta_ajustada}")

=== DEPOIS DO FINE-TUNING ===
Instrução: How do I activate cruise control?
Resposta ajustada: To use cruise control in a 2023 Subaru Outback:
1. Press the 'CRUISE' button on the steering wheel
2. Accelerate to desired speed (above 25 mph)
3. Press 'SET' to engage


## 📊 15. Comparação e Conclusão

- **Antes do fine-tuning:** o Llama-3.2-3B-Instruct responde de forma razoável mas genérica.
- **Depois do fine-tuning:** com LoRA sobre ~253 pares de neurociência, o modelo
  passa a responder com a profundidade e terminologia do material de referência.

### 📌 Comparativo entre os dois pipelines

| Parâmetro | Phi-4-mini-instruct | Llama-3.2-3B-Instruct |
|---|---|---|
| `MODEL_NAME` | `microsoft/Phi-4-mini-instruct` | `meta-llama/Llama-3.2-3B-Instruct` |
| `trust_remote_code` | `True` (obrigatório) | Não necessário |
| Chat template tokens | `<\|system\|>...<\|end\|>` | `<\|begin_of_text\|>...<\|eot_id\|>` |
| `target_modules` | Idênticos (mesma nomenclatura GQA) | Idênticos |
| Acesso HuggingFace | Aberto | Requer aceite de licença + token |
| `output_dir` | `lora_models/causal_model_1` | `lora_models/llama32_neuro` |

### 📌 Resumo dos conceitos-chave

| Conceito | Descrição |
|---|---|
| **Full fine-tuning** | Atualiza todos os pesos do modelo. |
| **PEFT / LoRA** | Atualiza apenas matrizes A e B de baixo rank — <1% dos parâmetros. |
| **QLoRA** | LoRA aplicado sobre modelo quantizado em 4-bit — economiza VRAM. |
| **`r`** | Rank da decomposição — controla capacidade da adaptação. |
| **`lora_alpha`** | Fator de escala: `alpha/r` determina a intensidade da atualização. |
| **`target_modules`** | Camadas onde os adaptadores LoRA são inseridos. |
| **Chat template** | Formato de tokens que cada modelo espera — varia entre arquiteturas. |
